# SF Case Files — Report Analysis

> *"The report is lying to you. I'm not sure how yet. But I will be."* — Adrian Monk

This notebook investigates the reports connected to the **SFCaseFiles** semantic model using Semantic Link Labs.

| Section | Theme | PBIR Required |
|---|---|---|
| Scene Survey | Inventory all reports and model connections | No |
| Cataloging the Evidence | Full ReportWrapper inspection | ✅ Yes |
| The Broken Evidence | Detect invalid field references | ✅ Yes |
| The Inspection | Report Best Practice Analyzer | ✅ Yes |
| The Field Report | Unused measures cross-reference | No |
| The Fix | Repair broken references + report actions | No / ✅ Yes |

---

⚠️ **PBIR Note:** Sections marked ✅ require reports to be in **PBIR format**.
PBIR is in preview but is becoming the default report format in Fabric — the future is already here.

---
## Setup

In [ ]:
%run 00_Setup_Fresh

`ReportWrapper` is the primary interface for report inspection throughout this notebook.
`REPORT` must be in **PBIR format** for all sections marked ✅ in the table above.
`readonly=True` is the default — we investigate first, act later.

In [ ]:
from sempy_labs.report import ReportWrapper

DATASET = "SFCaseFiles_ReportLab"
REPORT = "SFCaseFiles_ReportLab_Thin"

# Connect to the report — used throughout this notebook
rpt = ReportWrapper(report=REPORT, workspace=WORKSPACE)

---
## Scene Survey — Inventory
> *"Before I touch anything, I need to know what's here."*
Catalog all reports in the workspace and confirm which semantic model each one is connected to.
A report pointing at the wrong model — or a deleted model — is already broken.
No PBIR required for this section — `fabric.list_reports()` works on any report format.

In [ ]:
# All reports in the workspace
df_reports = fabric.list_reports(workspace=WORKSPACE)
display(df_reports)

In [ ]:
# Which reports are connected to SFCaseFiles_ReportLab specifically?
df_sf_reports = labs.list_reports_using_semantic_model(
    dataset=DATASET,
    workspace=WORKSPACE,
)

print(f"Reports connected to '{DATASET}': {len(df_sf_reports)}")
display(df_sf_reports)

---
## Cataloging the Evidence — ReportWrapper Inventory
> *"Monk catalogs everything. Every page. Every visual. Every filter. Every bookmark."*
A full structural inventory of the report using `ReportWrapper`.
Each cell below answers a specific question about what's inside the report.
Requires PBIR format.

In [ ]:
# Pages
print("=== Pages ===")
display(rpt.list_pages())

In [ ]:
# Visuals — one row per visual across all pages
print("=== Visuals ===")
display(rpt.list_visuals())

In [ ]:
# Visual objects — field-level breakdown of what each visual uses
print("=== Visual Objects ===")
display(rpt.list_visual_objects())

In [ ]:
# Custom visuals — are there any third-party visuals installed?
# Unused custom visuals add overhead with no benefit
print("=== Custom Visuals ===")
display(rpt.list_custom_visuals())

In [ ]:
# Bookmarks
print("=== Bookmarks ===")
display(rpt.list_bookmarks())

In [ ]:
# Visual interactions — which visuals cross-filter which?
print("=== Visual Interactions ===")
display(rpt.list_visual_interactions())

### Filter Audit
> *"Every filter is a constraint. Monk wants to know all the constraints."*
Filters exist at three tiers: report level (every page), page level (all visuals on a page),
and visual level (one visual only). Filters at the wrong tier are hard to find,
hard to maintain, and easy to forget — and nobody documents them.

In [ ]:
# Report-level filters — apply to every page
print("=== Report Filters ===")
display(rpt.list_report_filters())

In [ ]:
# Page-level filters — apply to all visuals on a page
print("=== Page Filters ===")
display(rpt.list_page_filters())

In [ ]:
# Visual-level filters — apply to a single visual only
print("=== Visual Filters ===")
display(rpt.list_visual_filters())

### Report-Level Measures
> *"Monk doesn't let witnesses keep evidence at home. It belongs in the file."*
Report-level measures live inside the report definition, not the semantic model.
They can't be reused across reports, they're invisible to BPA, and they don't survive
a report rebuild. If you find any here — they belong in the model.

In [ ]:
# Report-level measures — these should almost always live in the semantic model instead
print("=== Report-Level Measures ===")
df_rlm = rpt.list_report_level_measures()
display(df_rlm)

if df_rlm is not None and len(df_rlm) > 0:
    print(f"\n{len(df_rlm)} report-level measure(s) found.")
    print("   Consider migrating these to the semantic model for reuse and governance.")
else:
    print("No report-level measures found.")

## ✅ Broken Evidence: Check Visual References

Do the fields and measures used by our visuals still exist?

We’ll check references in visuals and their filters, including measures
defined in the report itself. Any invalid references need investigation.
In SLL 0.18.1, the combined `list_semantic_model_objects(extended=True)` check
validates measures against the shared model only. We use the visual-specific
methods here because they also recognize report-level measures.
A clear result applies to these checked references, not every possible report issue.

In [ ]:
# Check visual and visual-filter references, including report-level measures
# Use a fresh report definition after publishing any changes.
rpt = ReportWrapper(report=REPORT, workspace=WORKSPACE)
df_visuals = rpt.list_visual_objects(extended=True).assign(
    Reference="Visual"
)
df_filters = rpt.list_visual_filters(extended=True).assign(
    Reference="Visual Filter"
)

df_checked = pd.concat([df_visuals, df_filters], ignore_index=True)
df_broken = df_checked[
    df_checked["Valid Semantic Model Object"].eq(False)
]

validity = df_checked["Valid Semantic Model Object"]
if df_checked.empty or validity.isna().any() or not validity.isin([True, False]).all():
    print("Some reference checks are incomplete. Review the diagnostics below.")
    display(df_checked)
elif df_broken.empty:
    print("No invalid visual or visual-filter references found.")
else:
    print(f"Invalid references found: {len(df_broken)}")
    display(df_broken)

In [ ]:
# Inventory all referenced objects, including report and page filters.
# Validation is performed separately above, using report-aware visual checks.
df_smo = rpt.list_semantic_model_objects(extended=False)
display(df_smo)

In [ ]:
# Inventory objects used across reports connected to this semantic model.
# This is an inventory, not a report-aware broken-reference check in SLL 0.18.1.
df_all_smo = labs.list_report_semantic_model_objects(
    dataset=DATASET,
    workspace=WORKSPACE,
    extended=False,
)
display(df_all_smo)

---
## The Inspection — Report Best Practice Analyzer
> *"I have a list. I always have a list."*
The Report BPA evaluates the report structure against a set of rules covering
performance, error prevention, and formatting.
Start by viewing the default rules, then run the full inspection.
Requires PBIR format.
Treat BPA findings as investigation leads. If a model-object validity rule flags
a report-level measure, cross-check the report-aware Broken Evidence results
and the measure definition before calling it broken.

In [ ]:
# View the default report BPA rules
df_rules = rep.report_bpa_rules()
display(df_rules)

In [ ]:
# Run the Report BPA — HTML visualization
rep.run_report_bpa(report=REPORT, workspace=WORKSPACE)

### Custom Rules
The default rules cover general report hygiene. You can also define your own rules as a DataFrame
and pass them in via the `rules` parameter.
The example below adds two performance rules around visual count and visual object density.
Broken Evidence handles reference validity separately using report-aware methods.

In [ ]:
# Run two custom performance rules.
# Reference validation is handled in the Broken Evidence section.
import pandas as pd

custom_rules = pd.DataFrame(
    [
        (
            "Performance",
            "Page",
            "Warning",
            "Reduce the number of visible visuals on the page",
            lambda df: df["Visible Visual Count"] > 15,
            "Pages with many visible visuals may require more rendering and query work.",
            "",
        ),
        (
            "Performance",
            "Visual",
            "Warning",
            "Reduce the number of objects within visuals",
            lambda df: df["Visual Object Count"] > 5,
            "Review visuals with many fields and measures for unnecessary complexity.",
            "",
        ),
    ],
    columns=[
        "Category", "Scope", "Severity", "Rule Name",
        "Expression", "Description", "URL",
    ],
)

# Avoid the SLL 0.18.1 empty-results URL-column bug.
pages = rpt.list_pages()
visuals = rpt.list_visuals()

has_findings = (
    pages["Visible Visual Count"].gt(15).any()
    or visuals["Visual Object Count"].gt(5).any()
)

if has_findings:
    rep.run_report_bpa(
        report=REPORT,
        workspace=WORKSPACE,
        rules=custom_rules,
    )
else:
    print("Neither custom performance rule found any violations.")

### Persist the Results
Same pattern as the model BPA — `export=True` appends results to a delta table in the
attached lakehouse. Run it on a schedule alongside `run_model_bpa_bulk()` and you have
a full governance audit trail for both models and reports.

In [ ]:
# Export custom BPA findings to the attached Lakehouse.
# Run default report BPA rules and append findings to reportbparesults.
rep.run_report_bpa(
    report=REPORT,
    workspace=WORKSPACE,
    export=True,
)

---
## The Field Report — What's Actually Used?
> *"You'd be amazed what people put in reports that nobody ever looks at."*
Cross-reference every measure in the semantic model against what the connected reports
actually reference. Unused measures are either dead weight, candidates for cleanup,
or — more interesting — evidence that something broke silently and nobody noticed.
No PBIR required for this section.

In [ ]:
# All measures in the model
df_measures = fabric.list_measures(DATASET, workspace=WORKSPACE)

# All model objects used in reports connected to this model
df_used = labs.list_report_semantic_model_objects(
    dataset=DATASET,
    workspace=WORKSPACE,
    extended=True
)

used_measures = set(df_used[df_used["Object Type"] == "Measure"]["Object Name"].tolist())
all_measures  = set(df_measures["Measure Name"].tolist())
unused        = all_measures - used_measures

print(f"Total measures in model:       {len(all_measures)}")
print(f"Measures used in reports:      {len(used_measures)}")
print(f"Measures not used in reports:  {len(unused)}")

if unused:
    print("\nUnused measures:")
    for m in sorted(unused):
        print(f"  - {m}")

---
## The Fix — Actions and Repairs
> *"Now that I know what's wrong, I can fix it. I always fix it."*
Two categories of fixes:
1. **Repair** — fix broken visual references with `lineage_view` in **notebook 6, Case File: The Broken Reference**
2. **Housekeeping** — report actions that clean up, standardize, and optimize the report
⚠️ Everything in this section writes changes back to the service. Review before running.

### Repairing Broken Field References
The repair lives in **6.SFCaseFiles_LineageRepair**: inspect the broken reference, map it to the correct measure in `lineage_view`, save, then rerun the check and reopen the visual.
The earlier whole-report find-and-replace pattern was removed — it could change unrelated text and never proved the visual worked afterwards.

### Report Housekeeping Actions

These functions write changes back to the service.
Uncomment and run as needed — all require PBIR format.

In [ ]:
# Apply a custom theme (from lakehouse or URL)
# rpt.set_theme(theme_file_path='/lakehouse/default/Files/SFCaseFiles_Theme.json')
# rpt.set_theme(theme_file_path='https://raw.githubusercontent.com/PowerBiDevCamp/FabricUserApiDemo/main/FabricUserApiDemo/DefinitionTemplates/Shared/Reports/StaticResources/SharedResources/BaseThemes/CY23SU08.json')

print("set_theme — uncomment to apply.")

In [ ]:
# Hide/show a page
# rpt.set_page_visibility(page_name='Tooltip', hidden=True)

# Set the active page (first page shown when report opens)
# rpt.set_active_page(page_name='Overview')

print("Page visibility / active page — uncomment to apply.")

In [ ]:
# Hide all tooltip and drillthrough pages (they shouldn't be visible in the nav)
# rpt.hide_tooltip_drillthrough_pages()

# Disable 'show items with no data' for all visuals (common performance drain)
# rpt.disable_show_items_with_no_data()

# Remove any custom visuals that aren't actually used in any visual on any page
# rpt.remove_unnecessary_custom_visuals()

print("Housekeeping actions — uncomment to apply.")

In [ ]:
# Rebind report to a different semantic model
# Useful when a model is renamed, promoted, or replaced by a new version
# rep.report_rebind(
#     report=REPORT,
#     dataset="SFCaseFiles_V2",
#     report_workspace=WORKSPACE,
#     dataset_workspace=WORKSPACE
# )

print("report_rebind — uncomment to apply.")

---

## Summary

| What | Function | PBIR Required |
|---|---|---|
| List all reports in workspace | `fabric.list_reports()` | No |
| Reports connected to a model | filter on `Dataset Name` | No |
| List pages | `rpt.list_pages()` | ✅ Yes |
| List visuals | `rpt.list_visuals()` | ✅ Yes |
| List visual objects (field-level) | `rpt.list_visual_objects()` | ✅ Yes |
| List custom visuals | `rpt.list_custom_visuals()` | ✅ Yes |
| List bookmarks | `rpt.list_bookmarks()` | ✅ Yes |
| List visual interactions | `rpt.list_visual_interactions()` | ✅ Yes |
| List report / page / visual filters | `rpt.list_report_filters()` etc. | ✅ Yes |
| Find report-level measures | `rpt.list_report_level_measures()` | ✅ Yes |
| Check visual / visual-filter references | `rpt.list_visual_objects(extended=True)` + `list_visual_filters(extended=True)` | ✅ Yes |
| Inventory references across connected reports | `labs.list_report_semantic_model_objects(extended=False)` | No |
| Report Best Practice Analyzer | `rep.run_report_bpa()` | ✅ Yes |
| Unused measures analysis | `list_measures` + `list_report_semantic_model_objects` | No |
| Apply theme | `rpt.set_theme()` | ✅ Yes |
| Page visibility / active page | `rpt.set_page_visibility()` / `set_active_page()` | ✅ Yes |
| Remove unused custom visuals | `rpt.remove_unnecessary_custom_visuals()` | ✅ Yes |
| Hide tooltip/drillthrough pages | `rpt.hide_tooltip_drillthrough_pages()` | ✅ Yes |
| Disable show items with no data | `rpt.disable_show_items_with_no_data()` | ✅ Yes |
| Repair broken field references | `lineage_view` — see notebook 6 | No |
| Rebind report to new model | `rep.report_rebind()` | No |

> *"You'll thank me later. You always do."* — Adrian Monk

---

### Resources
- [Semantic Link Labs GitHub](https://github.com/microsoft/semantic-link-labs)
- [Report Analysis helper notebook](https://github.com/microsoft/semantic-link-labs/blob/main/notebooks/Report%20Analysis.ipynb)
- [sempy_labs.report docs](https://semantic-link-labs.readthedocs.io/en/stable/sempy_labs.report.html)
- [datashepherd.dev](https://datashepherd.dev)